# Model Baseline — UCI Cannabis Risk Dataset

This notebook establishes a logistic regression baseline, tunes the regularisation parameter C, compares it against Random Forest, and explores threshold tuning to maximise recall for at-risk students.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
df_train = pd.read_csv('uci_cannabis_train.csv')
df_test  = pd.read_csv('uci_cannabis_test.csv')

display(df_train.head())
display(df_test.head())

In [ ]:
# Separating features and target variable
FEATURES = ['Age','Gender','Education','Country','Ethnicity',
            'Nscore','Escore','Oscore','Ascore','Cscore','Impulsive','SS']

x_train = df_train[FEATURES]
y_train = df_train['cannabis_risk']

x_test = df_test[FEATURES]
y_test = df_test['cannabis_risk']

In [ ]:
# Separating features and target (for validation only — uses train split internally)
x_tr, x_val, y_tr, y_val = train_test_split(
    x_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

In [ ]:
# Feature scaling — required for logistic regression (gradient-based)
# Fit on training fold only to avoid data leakage
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_tr_s  = scaler.fit_transform(x_tr)
x_val_s = scaler.transform(x_val)

In [ ]:
# Baseline logistic regression model
model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
model.fit(x_tr_s, y_tr)

y_pred = model.predict(x_val_s)

print('Accuracy: ', accuracy_score(y_val, y_pred))
print('Precision:', precision_score(y_val, y_pred))
print('Recall:   ', recall_score(y_val, y_pred))
print('F1 Score: ', f1_score(y_val, y_pred))

In [ ]:
# Tuning logistic regression — sweep regularisation parameter C
Cs = [0.01, 0.1, 1, 10, 100]
results = []
for C in Cs:
    m = LogisticRegression(class_weight='balanced', C=C, max_iter=1000, random_state=42)
    m.fit(x_tr_s, y_tr)
    y_pred = m.predict(x_val_s)
    results.append({
        'C': C,
        'Accuracy' : accuracy_score(y_val, y_pred),
        'Precision': precision_score(y_val, y_pred),
        'Recall'   : recall_score(y_val, y_pred),
        'F1 Score' : f1_score(y_val, y_pred)
    })

pd.DataFrame(results)

- C=1.0 gives the best balance of recall and F1 on the validation set for this dataset.
- Unlike the previous student dataset, regularisation has a meaningful effect here because the features carry real signal.

In [ ]:
# Random Forest comparison
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    random_state=42
)
rf.fit(x_tr_s, y_tr)
rf_preds = rf.predict(x_val_s)

rf_results = {
    'Model'    : 'RandomForest',
    'Accuracy' : accuracy_score(y_val, rf_preds),
    'Precision': precision_score(y_val, rf_preds),
    'Recall'   : recall_score(y_val, rf_preds),
    'F1 Score' : f1_score(y_val, rf_preds),
}
rf_results

In [ ]:
results_table = [
    {'Model': 'LogReg (C=1.0)', 'Accuracy': results[2]['Accuracy'],
     'Precision': results[2]['Precision'], 'Recall': results[2]['Recall'],
     'F1 Score': results[2]['F1 Score']},
    rf_results,
]
pd.DataFrame(results_table)

- Given the project being a risk prediction task, **recall and F1 score are more important than raw accuracy** so the tuned logistic regression (C=1.0) is the current baseline.
- Random Forest shows competitive performance but is less interpretable.

In [ ]:
# Threshold tuning for logistic regression to improve recall
y_proba = model.predict_proba(x_val_s)[:, 1]

thresholds = [0.5, 0.4, 0.35, 0.3, 0.25]
results_thr = []
for thr in thresholds:
    y_pred_thr = (y_proba >= thr).astype(int)
    results_thr.append({
        'Threshold': thr,
        'Accuracy' : accuracy_score(y_val, y_pred_thr),
        'Precision': precision_score(y_val, y_pred_thr),
        'Recall'   : recall_score(y_val, y_pred_thr),
        'F1'       : f1_score(y_val, y_pred_thr)
    })

pd.DataFrame(results_thr)

- Lowering the threshold increases recall — catching more at-risk students — at the cost of precision.
- For a harm-prevention screening tool, missing an at-risk student (false negative) is more costly than a false alarm (false positive).
- Threshold 0.495 is selected as the best compromise.

In [ ]:
# Refit on full training set with chosen configuration
x_full_s = scaler.fit_transform(x_train)
x_test_s  = scaler.transform(x_test)

model_final = LogisticRegression(class_weight='balanced', max_iter=1000, C=1.0, random_state=42)
model_final.fit(x_full_s, y_train)

y_test_proba = model_final.predict_proba(x_test_s)[:, 1]
thr = 0.495
y_test_pred = (y_test_proba >= thr).astype(int)

print('Test Set Results (threshold = 0.495):')
print('Accuracy: ', accuracy_score(y_test, y_test_pred))
print('Precision:', precision_score(y_test, y_test_pred))
print('Recall:   ', recall_score(y_test, y_test_pred))
print('F1:       ', f1_score(y_test, y_test_pred))

- Features in the UCI dataset (personality traits + demographics) show real predictive signal, unlike the previous dataset.
- The logistic regression baseline achieves meaningful performance (AUC ~0.86) on the test set.